In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# Direct 5-Class Random Forest Model for ESI 1 to 5 (`models/rf_direct_esi1to5.ipynb`)

This notebook trains a **Direct 5-Class Random Forest Model** to straight up predict **ESI Level 1, 2, 3, 4, and 5** using **35 Predictor Features** (19 raw inputs + 16 continuous vital delta & range features; strictly excluding binary vital anomaly flags), with **Targeted Bootstrap Oversampling Specifically for ESI 1 and ESI 5 Rows** on the training set:

### System Architecture & Workflow
1. **Direct 5-Class Target (`"1"`, `"2"`, `"3"`, `"4"`, `"5"`)**:
   - Straight up multiclass prediction across all 5 Emergency Severity Index tiers.
2. **Predictor Feature Selection (35 Total Features)**:
   - **19 Raw Inputs**: `age`, `gender`, `cc_breathingdifficulty`, `triage_vital_hr`, `triage_vital_sbp`, `triage_vital_rr`, `triage_vital_o2`, `pulse_last`, `resp_last`, `spo2_last`, `sbp_last`, `pulse_min`, `resp_min`, `spo2_min`, `sbp_min`, `pulse_max`, `resp_max`, `spo2_max`, `sbp_max`.
   - **16 Continuous Vital Delta & Range Features**: `hr_mean_to_last`, `sbp_mean_to_last`, `spo2_mean_to_last`, `rr_mean_to_last`, `hr_range`, `rr_range`, `spo2_range`, `sbp_range`, `hr_last_to_min`, `rr_last_to_min`, `spo2_last_to_min`, `sbp_last_to_min`, `hr_last_to_max`, `rr_last_to_max`, `spo2_last_to_max`, `sbp_last_to_max`.
3. **Targeted Oversampling for ESI 1 and ESI 5**:
   - Applies bootstrap oversampling specifically to minority extreme tiers (**ESI 1** and **ESI 5**) on the training set prior to model fitting.
4. **Single Validation Split Partitioning (From Config)**:
   - Stratified Partitioning into Train, Validation (`val_size`), and Test (`test_size`) sets parsed directly from `config/triage_conf.json`.
5. **5-Class Random Forest Model (`ranger`)**: Fits decision trees with `probability = TRUE`.
6. **Targeted Benchmarking Suite**: Evaluates ONLY **Recall (Sensitivity)**, **Specificity**, **Balanced Accuracy**, and **ROC-AUC**.
7. **Reports & Artifacts**:
   - **CSV Reports**: `reports/rf_direct_esi1to5_val_report.csv` and `reports/rf_direct_esi1to5_test_report.csv`.
   - **Model Export**: Saved to `deploy/rf_direct_esi1to5_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(ranger)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data & Construct 35 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
raw_df <- get(data_obj_name, envir = data_env)
target_col_name <- config$classes$target_col
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
get_vec <- function(col_name, default_val = 0) {
  if (col_name %in% names(raw_df)) {
    res <- raw_df[[col_name]]
    res[is.na(res)] <- default_val
    return(res)
  } else {
    return(rep(default_val, nrow(raw_df)))
  }
}
p_last   <- get_vec("pulse_last")
p_max    <- get_vec("pulse_max")
p_min    <- get_vec("pulse_min")
s_last   <- get_vec("sbp_last")
s_max    <- get_vec("sbp_max")
s_min    <- get_vec("sbp_min")
o2_last  <- get_vec("spo2_last")
o2_max   <- get_vec("spo2_max")
o2_min   <- get_vec("spo2_min")
r_last   <- get_vec("resp_last")
r_max    <- get_vec("resp_max")
r_min    <- get_vec("resp_min")
t_hr     <- get_vec("triage_vital_hr")
t_sbp    <- get_vec("triage_vital_sbp")
t_o2     <- get_vec("triage_vital_o2")
t_rr     <- get_vec("triage_vital_rr")
# Construct 35 Predictor Features
df_full <- data.frame(
  # 19 Raw Inputs
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  triage_vital_hr         = t_hr,
  triage_vital_sbp        = t_sbp,
  triage_vital_rr         = t_rr,
  triage_vital_o2         = t_o2,
  pulse_last              = p_last,
  resp_last               = r_last,
  spo2_last               = o2_last,
  sbp_last                = s_last,
  pulse_min               = p_min,
  resp_min                = r_min,
  spo2_min                = o2_min,
  sbp_min                 = s_min,
  pulse_max               = p_max,
  resp_max                = r_max,
  spo2_max                = o2_max,
  sbp_max                 = s_max,
  
  # 16 Continuous Vital Delta & Range Features
  hr_mean_to_last         = t_hr - p_last,
  sbp_mean_to_last        = t_sbp - s_last,
  spo2_mean_to_last       = t_o2 - o2_last,
  rr_mean_to_last         = t_rr - r_last,
  
  hr_range                = p_max - p_min,
  rr_range                = r_max - r_min,
  spo2_range              = o2_max - o2_min,
  sbp_range               = s_max - s_min,
  
  hr_last_to_min          = p_last - p_min,
  rr_last_to_min          = r_last - r_min,
  spo2_last_to_min        = o2_last - o2_min,
  sbp_last_to_min         = s_last - s_min,
  
  hr_last_to_max          = p_last - p_max,
  rr_last_to_max          = r_last - r_max,
  spo2_last_to_max        = o2_last - o2_max,
  sbp_last_to_max         = s_last - s_max
)
# Target Mapping: Direct 5-Class Target (1, 2, 3, 4, 5)
raw_esi <- as.character(raw_df[[target_col_name]])
df_full$target_col <- factor(raw_esi, levels = c("1", "2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows (Remaining: %d)\n", initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Direct 5-Class Dataset Ready (35 Features): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
print(table(df_full$target_col))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Partitioning & Targeted Oversampling Specifically for ESI 1 and ESI 5
# ---------------------------------------------------------
set.seed(config$training$random_state)
test_size <- config$training$test_size
val_size  <- config$training$val_size
in_train_val <- createDataPartition(df_full$target_col, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_col, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
oversample_esi1_and_esi5 <- function(df_train, target_count_multiplier = 5.0) {
  counts <- table(df_train$target_col)
  cat("Training set class distribution before oversampling:\n")
  print(counts)
  
  res_df <- df_train
  target_classes <- c("1", "5")
  
  for (cls in target_classes) {
    cls_cnt <- counts[[cls]]
    target_cnt <- round(cls_cnt * target_count_multiplier)
    if (target_cnt > cls_cnt) {
      extra_needed <- target_cnt - cls_cnt
      cls_subset   <- df_train[df_train$target_col == cls, ]
      sampled_extra <- cls_subset[sample(1:cls_cnt, size = extra_needed, replace = TRUE), ]
      res_df       <- rbind(res_df, sampled_extra)
    }
  }
  
  cat("Training set class distribution after oversampling ESI 1 and ESI 5:\n")
  print(table(res_df$target_col))
  return(res_df)
}
cat("Applying targeted oversampling for ESI 1 and ESI 5 rows...\n")
train_df_oversampled <- oversample_esi1_and_esi5(train_df, target_count_multiplier = 5.0)
binary_cols <- c("gender", "cc_breathingdifficulty")
cont_cols   <- setdiff(names(train_df), c(binary_cols, "target_col"))
preproc <- preProcess(train_df_oversampled[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_scaled <- predict(preproc, train_df_oversampled)
val_scaled   <- predict(preproc, val_df)
test_scaled  <- predict(preproc, test_df)
cat(sprintf("Dataset Splits: Train (Oversampled ESI 1 & 5) = %d, Val = %d, Test = %d\n",
            nrow(train_scaled), nrow(val_scaled), nrow(test_scaled)))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train Direct 5-Class Random Forest Model (35 Features)
# ---------------------------------------------------------
set.seed(config$training$random_state)
rf_direct_model <- ranger(
  formula     = target_col ~ .,
  data        = train_scaled,
  num.trees   = 250,
  probability = TRUE,
  seed        = config$training$random_state
)
cat("Direct 5-Class Random Forest Model Training Complete (35 Features, Oversampled ESI 1 & 5)!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Evaluate Direct 5-Class Performance on Validation & Test Sets (Recall, Specificity, BalAcc, ROC-AUC ONLY)
# ---------------------------------------------------------
eval_rf_direct_split <- function(df_split, split_name) {
  raw_probs <- predict(rf_direct_model, data = df_split)$predictions
  pred_idx  <- apply(raw_probs, 1, which.max)
  pred_fac  <- factor(colnames(raw_probs)[pred_idx], levels = c("1", "2", "3", "4", "5"))
  act_fac   <- factor(df_split$target_col, levels = c("1", "2", "3", "4", "5"))
  
  cm   <- confusionMatrix(pred_fac, act_fac)
  
  rec_by_class     <- as.numeric(cm$byClass[, "Sensitivity"])
  spec_by_class    <- as.numeric(cm$byClass[, "Specificity"])
  bal_acc_by_class <- as.numeric(cm$byClass[, "Balanced Accuracy"])
  
  rec_by_class[is.na(rec_by_class)]         <- 0
  spec_by_class[is.na(spec_by_class)]       <- 0
  bal_acc_by_class[is.na(bal_acc_by_class)] <- 0
  
  roc_auc_by_class <- sapply(1:5, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    r_obj    <- tryCatch(pROC::roc(act_bin, raw_probs[, i]), error = function(e) NULL)
    if (!is.null(r_obj)) as.numeric(r_obj$auc) else NA
  })
  macro_rec     <- mean(rec_by_class)
  macro_spec    <- mean(spec_by_class)
  macro_bal_acc <- mean(bal_acc_by_class)
  macro_auc     <- mean(roc_auc_by_class, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   DIRECT 5-CLASS RANDOM FOREST (OVERSAMPLED) - %s BENCHMARK\n", toupper(split_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Macro Recall (Sens) : %.4f\n", macro_rec))
  cat(sprintf("  Macro Specificity   : %.4f\n", macro_spec))
  cat(sprintf("  Macro Balanced Acc  : %.4f\n", macro_bal_acc))
  cat(sprintf("  Macro ROC-AUC       : %.4f\n", macro_auc))
  cat(sprintf("============================================================\n\n"))
  print(cm$table)
  cat("\n\n")
  
  return(data.frame(
    Split = split_name,
    Macro_Recall           = round(macro_rec, 4),
    Macro_Specificity      = round(macro_spec, 4),
    Macro_Balanced_Accuracy= round(macro_bal_acc, 4),
    Macro_ROC_AUC          = round(macro_auc, 4)
  ))
}
val_rep  <- eval_rf_direct_split(val_scaled,  "Validation")
test_rep <- eval_rf_direct_split(test_scaled, "Test")
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(val_rep,  file = file.path(reports_dir, "rf_direct_esi1to5_val_report.csv"),  row.names = FALSE)
write.csv(test_rep, file = file.path(reports_dir, "rf_direct_esi1to5_test_report.csv"), row.names = FALSE)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Save Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) deploy_dir <- "deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
saveRDS(list(model = rf_direct_model, preproc = preproc), file = file.path(deploy_dir, "rf_direct_esi1to5_model.rds"))
cat("Direct 5-Class Random Forest Model (Oversampled ESI 1 & 5) saved to deploy/rf_direct_esi1to5_model.rds\n")